In [2]:
import pandas as pd
import numpy as np
import openpyxl
import re

In [130]:
import os
from tqdm import tqdm
from time import sleep
import math

to-do:

 ~~выделить частые ключевые слова в кейсах из прошлого проекта (пр., медсестра, рождение и т.д.)~~

~~обновить словарь с поиском ключевых слов, снова прогнать тексты через это~~

## Упоминаются ли акушерки и больницы в приговорах?

### Приводим данные в машиночитаемый вид, удаляем пустые приговоры

In [43]:
# path = "/Users/Helen/Desktop"
df = pd.read_csv("/Users/Helen/Downloads/macOS-x64.sudrfscraper/results/childbirth_all_articles.csv",
                sep=';',
                on_bad_lines='skip')

In [44]:
# pd.set_option('display.max_colwidth', None)
# # df[df['cui'] == '47RS0001-01-2024-001637-05'][['names', 'court_name', 'entryDate', 'caseNumber']]
# # df[df['caseNumber'] == '1-28/2025'].sort_values(by='cui')
# # df['case_uid'] = df['link_meta'].str.extract(r"&case_uid=(.+)&")
# # df = df.drop_duplicates(subset='link_meta')
# df['link_meta'].value_counts()

In [45]:
# преобразуем данные в колонке 'text' в строки (были объектами)
df['text'] = df['text'].astype('string')

# выделяем подсет данных для более быстрой обработки
new_df = df[['id', 'text', 'link_text']]

# удаляем строки, где текст заседаний короче 10 символов
new_df = new_df[new_df['text'].str.len() >= 2]

In [46]:
# приводим тексты заседаний в прописные буквы, удаляем все не-буквы
new_df['text'] = new_df['text'].str.lower()
new_df['text'] = new_df['text'].str.replace(r'[^а-яА-ЯёЁ\s]', ' ', regex=True)
new_df['text'] = new_df['text'].str.replace(r'\s+', ' ', regex=True)

In [47]:
# создаём предварительный фильтр, чтобы отобрать дела, в которых упоминается война
# новый фильтр после чтения судактов, от 24/04/2026
rough_words = [
            r'\bакушер(о?)к',
            r'врач.*?(-?)акушер(.*?)\b',
            r"беременн",
            r"родоразрешение",
            r"акушер(.{0,3}?)(-?)гинеколог"
]

rough_pattern = re.compile('|'.join(rough_words), re.IGNORECASE)

# # функция, которая проверяет тексты приговоров по ключевым словам
def is_relevant(text):
    if text is None:  # Check if tokens is None
        return "!"
    # if 'покушение' in text.lower():
        # return None
    return 'yes' if rough_pattern.search(text) else None

df['pre_relevant'] = df['text'].apply(is_relevant)

a = len(df.loc[df['pre_relevant'] == 'yes', ['id']])
b = len(df.loc[df['text'].str.len() > 1, ['id']])
print(f"Всего нашлось {a} релевантных приговоров из {b} подходящих дел.")


Всего нашлось 51 релевантных приговоров из 58 подходящих дел.


In [48]:
# pd.set_option('display.max_colwidth', None)
# # df.loc[df['pre_relevant'] == 'yes', ['link_text']]
# df[df['pre_relevant'] == 'yes']['link_text']

In [49]:
df.to_csv('../output/articles_filtered_2.csv')

## Сравниванию судакты с и без 124 статьи

In [50]:
# загружаю отсмотренные судакты 
gsheetkey = "1Pcjjxsp4rm7WKJCzbd-EpVjMR4rKFWXNBmIL0HN_bUc"
# https://docs.google.com/spreadsheets/d/1Pcjjxsp4rm7WKJCzbd-EpVjMR4rKFWXNBmIL0HN_bUc/edit?usp=sharing

#sheet name
sheet_name = 'articles_filtered'

url=f'https://docs.google.com/spreadsheet/ccc?key={gsheetkey}&output=xlsx'
no_124 = pd.read_excel(url,sheet_name=sheet_name)

In [51]:
df['pre_relevant'].value_counts()


pre_relevant
yes    51
Name: count, dtype: int64

In [52]:
a = no_124[no_124['relevant'].notnull()]['id'].tolist()

## Объединение старой и новой баз данных

In [ ]:
# # загружаю новые судакты
# gsheetkey = "1Pcjjxsp4rm7WKJCzbd-EpVjMR4rKFWXNBmIL0HN_bUc"
# # https://docs.google.com/spreadsheets/d/1Pcjjxsp4rm7WKJCzbd-EpVjMR4rKFWXNBmIL0HN_bUc/edit?usp=sharing

# #sheet name
# sheet_name = 'articles_filtered'

# url=f'https://docs.google.com/spreadsheet/ccc?key={gsheetkey}&output=xlsx'
# new_articles = pd.read_excel(url,sheet_name=sheet_name)

In [100]:
# загружаю старые судакты
gsheetkey = "1OXN-pshdQUgM0rKwyy7r0jxcdHfTtgcNQk6ejgD6RuM"
# https://docs.google.com/spreadsheets/d/1OXN-pshdQUgM0rKwyy7r0jxcdHfTtgcNQk6ejgD6RuM/edit?usp=sharing

#sheet name
sheet_name = 'childbirth_all_articles'

url=f'https://docs.google.com/spreadsheet/ccc?key={gsheetkey}&output=xlsx'
old_articles = pd.read_excel(url,sheet_name=sheet_name)

In [103]:
old_articles.columns

Index(['id', 'region_name', 'region', 'instance', 'court_name', 'caseNumber',
       'entryDate', 'names', 'judge', 'resultDate', 'decision', 'endDate',
       'consideredBy', 'cui', 'link_meta', 'link_text', 'link_text_and_meta',
       'connected_links_1', 'connected_links_2', 'connected_links_3',
       'connected_links_4', 'text', 'relevant', 'mother killed',
       'child killed', 'no. of suspects', 'sentence years', 'drop reason',
       'commentary', 'pre_relevant', 'yearDate'],
      dtype='object')

In [104]:
old_articles['entryDate'] = pd.to_datetime(old_articles['entryDate'], errors='coerce')
old_articles['resultDate'] = pd.to_datetime(old_articles['resultDate'], errors='coerce')
old_articles['endDate'] = pd.to_datetime(old_articles['endDate'], errors='coerce')

/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_24837/683952475.py:3: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  old_articles['resultDate'] = pd.to_datetime(old_articles['resultDate'], errors='coerce')
/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_24837/683952475.py:4: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  old_articles['endDate'] = pd.to_datetime(old_articles['endDate'], errors='coerce')


count                              156
mean     2022-05-07 19:50:46.153846272
min                2019-04-04 00:00:00
25%                2020-12-26 18:00:00
50%                2022-05-30 00:00:00
75%                2023-11-15 12:00:00
max                2025-02-27 00:00:00
Name: resultDate, dtype: object

In [110]:
old_articles['resultDate'].describe()

count                              156
mean     2022-05-07 19:50:46.153846272
min                2019-04-04 00:00:00
25%                2020-12-26 18:00:00
50%                2022-05-30 00:00:00
75%                2023-11-15 12:00:00
max                2025-02-27 00:00:00
Name: resultDate, dtype: object

In [56]:
# old_articles = old_articles.loc[old_articles['relevant'] == 1]
# new_articles = new_articles.loc[new_articles['relevant'] == 1]
# all_articles = pd.concat([old_articles, new_articles])
# # all_articles

In [57]:
# all_articles['child killed'].isnull().values.any()

In [58]:
# # convert columns 'relevant', 'mother killed', 'child killed', 'no. of suspects' to int
# cols = ['relevant', 'mother killed', 'child killed', 'no. of suspects']
# # for i in cols:
# #     all_articles[i].fillna(np.nan)
# #     cols = ['relevant', 'mother killed', 'child killed', 'no. of suspects']
# for i in cols:
#     all_articles[i] = all_articles[i].astype('Int64')
#     print(i, all_articles[i].dtype)


In [59]:
# all_articles

In [60]:
# all_articles.to_csv("../output/obstetric_violence_2018_2026_2.csv")

# Описательная статистика

### Обновление датасета

Добавлено продолжительность рассмотрения дела, dummy-переменные о статьях

In [115]:
# # all_articles['article_number'] = all_articles['names'].str.findall(r'ст\.\s*(\d+)\s*ч\.\s*(\d+)')
# all_articles['109'] = all_articles['names'].str.contains(r'ст\.\s*109', regex=True)
# all_articles['118'] = all_articles['names'].str.contains(r'ст\.\s*118', regex=True)
# all_articles['124'] = all_articles['names'].str.contains(r'ст\.\s*124', regex=True)
# all_articles['238'] = all_articles['names'].str.contains(r'ст\.\s*238', regex=True)
# all_articles['293'] = all_articles['names'].str.contains(r'ст\.\s*293', regex=True)

In [87]:
# # how much time has passed between entryDate and resultDate in all_articles, and save it in a new column 'time_to_decision' in days
# all_articles['entryDate'] = pd.to_datetime(all_articles['entryDate'], errors='coerce')
# all_articles['resultDate'] = pd.to_datetime(all_articles['resultDate'], errors='coerce')
# all_articles['time_to_decision'] = (all_articles['resultDate'] - all_articles['entryDate']).dt.days

/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_24837/3750129592.py:2: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  all_articles['entryDate'] = pd.to_datetime(all_articles['entryDate'], errors='coerce')
/var/folders/5r/7bhf5tqj0wb9bbk62kmr6wk80000gp/T/ipykernel_24837/3750129592.py:3: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  all_articles['resultDate'] = pd.to_datetime(all_articles['resultDate'], errors='coerce')


In [117]:
# all_articles.to_csv('../output/obstetric_violence_2018_2026_updated.csv')

### Работа с обновленным датасетом

In [86]:
# загружаю названия регионов с кодами ОКАТО 
#get spreadsheets key from url
gsheetkey = "1kG1KznHiZk31tCdIY9jTALhVcv8GZw1dtj1mG46MZV0"
# https://docs.google.com/spreadsheets/d/1kG1KznHiZk31tCdIY9jTALhVcv8GZw1dtj1mG46MZV0/edit?usp=sharing

#sheet name
sheet_name = 'obstetric_violence_2018_2026_updated'

url=f'https://docs.google.com/spreadsheet/ccc?key={gsheetkey}&output=xlsx'
all_articles = pd.read_excel(url,sheet_name=sheet_name)

In [88]:
all_articles.columns

Index([' ', 'id', 'region_name', 'region', 'instance', 'court_name',
       'caseNumber', 'entryDate', 'names', 'judge', 'resultDate', 'decision',
       'endDate', 'consideredBy', 'cui', 'link_meta', 'link_text',
       'link_text_and_meta', 'connected_links_1', 'connected_links_2',
       'connected_links_3', 'connected_links_4', 'text', 'relevant',
       'mother killed', 'child killed', 'no. of suspects', 'sentence years',
       'final decision', 'commentary', 'pre_relevant', 'yearDate',
       'time_to_decision'],
      dtype='object')

In [99]:
# all_articles.groupby(by='drop reason')['time_to_decision'].describe()
all_articles[all_articles['final decision'] == 'ограничение свободы']['region_name'].value_counts()
# all_articles['region_name'].value_counts()

region_name
Архангельская область    2
Республика Дагестан      1
Красноярский край        1
Приморский край          1
Астраханская область     1
Иркутская область        1
Самарская область        1
Забайкальский край       1
Name: count, dtype: int64

In [ ]:
all_articles[all_articles['decision'] == 'Вынесен ПРИГОВОР']['drop reason']

1     истечение сроков
2           оправдание
5                  NaN
7     истечение сроков
8           оправдание
10    истечение сроков
13                 NaN
15    истечение сроков
16                 NaN
17    истечение сроков
19                 NaN
20                 NaN
21                 NaN
22          оправдание
27                 NaN
33                 NaN
35                 NaN
38                 NaN
39    истечение сроков
43          оправдание
44                 NaN
46    истечение сроков
47    истечение сроков
48    истечение сроков
50                 NaN
52    истечение сроков
53    истечение сроков
Name: drop reason, dtype: object

In [ ]:
all_articles[all_articles['time_to_decision'] == 6]['link_text']

3    http://iglinsky--bkr.sudrf.ru/modules.php?name=sud_delo&srv_num=1&name_op=doc&number=157187885&delo_id=1540006&new=0&text_number=1
Name: link_text, dtype: object

In [180]:
all_articles['entryDate'].describe()

count                               56
mean     2022-01-19 12:25:42.857142784
min                2019-01-21 00:00:00
25%                2020-04-06 12:00:00
50%                2021-12-03 12:00:00
75%                2023-05-10 12:00:00
max                2025-12-25 00:00:00
Name: entryDate, dtype: object

### Сколько новых дел

In [181]:
# filter articles with resultDate after 01/01/2024
new_articles = all_articles[all_articles['resultDate'] >= '2025-02-27']
ol_articles = all_articles[all_articles['resultDate'] < '2025-02-27']
new_articles['entryDate'].describe()

count                      8
mean     2025-04-04 21:00:00
min      2024-09-30 00:00:00
25%      2024-12-25 18:00:00
50%      2025-04-23 12:00:00
75%      2025-06-13 00:00:00
max      2025-12-25 00:00:00
Name: entryDate, dtype: object

In [182]:
old_articles['entryDate'].describe()

count                              156
mean     2021-10-31 07:23:04.615384576
min                2019-01-21 00:00:00
25%                2020-05-20 12:00:00
50%                2021-10-02 12:00:00
75%                2023-04-28 00:00:00
max                2025-01-31 00:00:00
Name: entryDate, dtype: object

In [114]:
# print region_name in new_articles and not in ol_articles
new_regions = set(new_articles['region_name'].unique())
old_regions = set(ol_articles['region_name'].unique())
print("Regions in new_articles but not in ol_articles:")
print(new_regions - old_regions)

Regions in new_articles but not in ol_articles:
{'Архангельская область', 'Липецкая область'}


In [112]:
new_articles['final decision'].value_counts()

final decision
истечение сроков              3
ограничение свободы           2
примирение                    1
условный срок                 1
дело вернули в прокуратуру    1
Name: count, dtype: int64

In [ ]:
# bar graph showing the distribution of time_to_decision in all_articles
from typing import Literal
import matplotlib.pyplot as plt

plt.hist(all_articles['time_to_decision'], bins=30)
plt.xlabel('Time to Decision (days)')
plt.ylabel('Frequency')
plt.title('Distribution of Time to Decision')
plt.show()

ImportError: cannot import name 'Literal' from 'pyparsing' (unknown location)

# Сравниваю сроки с данными Судебного департамента

In [124]:
# extract all digits which have 'ст.' in 'names' in all_articles
# all_articles = pd.read_csv('../output/obstetric_violence_2018_2026.csv')
all_articles['article_number'] = all_articles['names'].str.findall(r'ст\.\s*(\d+)\s*ч\.\s*(\d+)')
# all_articles['article_number'] = all_articles['article_number'].astype('Int64') 
all_articles.head(3)

,,id,region_name,region,instance,court_name,caseNumber,entryDate,names,judge,...,commentary,pre_relevant,yearDate,time_to_decision,109,118,124,238,293,article_number
0,138,149,Республика Башкортостан,2,FIRST,Кировский районный суд города Уфы Республики Башкортостан,1-376/2019,2019-07-16,Салахов Руслан Дамирович - ст.109 ч.2 УК РФ,Бикчурин А.Х,...,"сроки давности, п. «а» ч.1 ст. 78 УК РФ",NaN,2019,35,True,False,False,False,False,"[(109, 2)]"
1,150,338,Республика Башкортостан,2,FIRST,Иглинский межрайонный суд Республики Башкортостан,1-151/2019,2019-06-17,Рамазанова Лилия Иргалиевна - ст.109 ч.2 УК РФ,Залов А.Ф.,...,"1 год ограничения свободы, но освбождена за истечением срока давности",NaN,2019,80,True,False,False,False,False,"[(109, 2)]"
2,153,51,Республика Башкортостан,2,FIRST,Кировский районный суд города Уфы Республики Башкортостан,"1-17/2021 (1-466/2020,)",2020-10-23,Мухамадиева Марина Валентиновна - ст.293 ч.2 УК РФ,Ишкубатов М.М,...,ОПРАВДАНА,NaN,2021,257,False,False,False,False,True,"[(293, 2)]"


In [129]:
# calculate the most common article numbers in all_articles
all_articles['article_number'].value_counts()

# most common tuple in all_articles['article_number'], NOT A LIST
# all_articles['article_number'].value_counts().head(10).index[0]

article_number
[(109, 2)]                                  36
[(109, 2), (109, 2)]                         2
[(293, 1)]                                   2
[(118, 2), (109, 2)]                         2
[(293, 2)]                                   2
[(124, 2), (124, 2)]                         1
[(118, 2)]                                   1
[(118, 2), (111, 2)]                         1
[(238, 2), (238, 2), (238, 2)]               1
[(293, 2), (293, 2), (109, 2)]               1
[(33, 3), (292, 1), (292, 1), (109, 2)]      1
[(238, 2), (238, 2), (238, 2), (238, 2)]     1
[(109, 2), (109, 2), (109, 2)]               1
[(238, 1)]                                   1
[(238, 1), (115, 2)]                         1
[(327, 4), (118, 2), (327, 4), (118, 2)]     1
[(118, 2), (109, 2), (118, 2), (109, 2)]     1
Name: count, dtype: int64

In [40]:
all_articles['article_number'].isin(['109']).sum()

0

In [234]:
years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
ab = '/Users/Helen/Downloads/data_suddep_archive_165_v20260424/2025 год'
print(ab.split(" ")[-2])
# print(ab.split("/")[-1].split(" ")[-4])

/Users/Helen/Downloads/data_suddep_archive_165_v20260424/2025


### Собираю данные из отчетов Суд. деп-а

In [539]:
table.sort_values(by='year')

,year,article,total,restriction,suspention,disqualification,disq_plus,acquital,reconciliation,total_category,...,exemption (category),sent_back (category),sent_down (category),restriction_share,suspension_share,disq_plus_share,acquital_share,reconciliation_share,exemption_share,excuse_share
5,2018.0,109 ч. 2,151,115,2,0,0,39,42,1930,...,41,102,50,0.761589,0.013245,0.000000,0.258278,0.278146,0.021244,0.030570
4,2019.0,109 ч. 2,143,99,1,0,48,20,32,1904,...,50,115,49,0.692308,0.006993,0.335664,0.139860,0.223776,0.026261,0.039391
0,2020.0,109 ч. 2,114,82,1,0,33,18,16,1785,...,45,117,39,0.719298,0.008772,0.289474,0.157895,0.140351,0.025210,0.044258
1,2021.0,109 ч. 2,122,87,4,0,24,13,22,1764,...,47,106,22,0.713115,0.032787,0.196721,0.106557,0.180328,0.026644,0.054422
3,2022.0,109 ч. 2,116,74,0,0,21,12,37,1800,...,70,60,23,0.637931,0.000000,0.181034,0.103448,0.318966,0.038889,0.060000
2,2023.0,109 ч. 2,116,80,0,0,23,8,27,1558,...,67,44,20,0.689655,0.000000,0.198276,0.068966,0.232759,0.043004,0.049422
6,2024.0,109 ч. 2,124,80,2,0,15,9,10,1576,...,74,40,23,0.645161,0.016129,0.120968,0.072581,0.080645,0.046954,0.054569
7,2025.0,109 ч. 2,57,44,0,0,13,9,5,595,...,23,16,9,0.771930,0.000000,0.228070,0.157895,0.087719,0.038655,0.070588


In [540]:
over_the_folder_path = '/Users/Helen/Downloads/data_suddep_archive_165_v20260424'
years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024']
# article -> (df1_row, df2_row, df3_row, df4_row)
# article_rows = {
    # '109 ч. 2': (16, 11, 11, 10),
    # '118 ч. 2': (47, 15, 15, 12),
    # '124 ч. 2': (64,),
    # '238 ч. 1': (631, ),
    # '238 ч. 2': (632, ),
    # '293 ч. 1': (894, )
# }
# articles = ['109 ч. 2', '118 ч. 2', '124 ч. 2', '238 ч. 1', '238 ч. 2', '293 ч. 1']
# создаем пустой список, чтобы сложить туда данные
rows = []

In [541]:
def note_data(df0, df, df2, df3, df4):
    # if article not in article_rows:
    #     raise ValueError(f"Unknown article: '{article}'. Add it to article_rows.")
    # 16, r2, r3, r4 = article_rows[article]
    new_row = pd.Series({
        'year': df0.iloc[4, 8], #год
        'article': df.iloc[16, 0], #статья
        'total': df.iloc[16, 2], #всего осуждено
        'restriction': df.iloc[16, 8], #ограничение свободы
        'suspention': df.iloc[16, 5], #условный срок
        'disqualification': df.iloc[16, 13], #запрет на проф. деятельность
        'disq_plus': df.iloc[16, 31], #запрет на проф. деятельность как доп. наказание
        'acquital': df.iloc[16, 22], #оправдание
        'reconciliation': df.iloc[16, 25], #примирение сторон
        'total_category': df2.iloc[11, 3], #число лиц, против кот. есть судакт по виду преступления (иное посягательство на жизнь)
        'excuse (category)': df2.iloc[11, 34], #дело прекратили из-за истечения срока давности
        'exemption (category)': df3.iloc[11, 26], #освободили от наказания из-за истечения срока давности
        'sent_back (category)': df4.iloc[10, 10], #дело вернули в прокуратуру
        'sent_down (category)': df4.iloc[10, 11] #дело вернули по подсудности
        })
    return new_row

In [542]:
for folder in os.listdir(over_the_folder_path):
    subfolder_path = os.path.join(over_the_folder_path, folder)
    a = folder.split(" ")
    if folder == '.DS_Store' and folder.endswith('.pdf'):
        continue
    elif len(a)<2: 
        continue
    elif (a[-2] in [str(y) for y in years]):               
        # достаем нужные листы в экселевском файле
        df0 = pd.read_excel(f'{over_the_folder_path}/{a[-2]} год/K7-{a[-2]}.xls')
        df = pd.read_excel(f'{over_the_folder_path}/{a[-2]} год/K7-{a[-2]}.xls', 
                                sheet_name='Раздел 1')
        df2 = pd.read_excel(f'{over_the_folder_path}/{a[-2]} год/K6-{a[-2]}.xls', 
                        sheet_name='Раздел 1')
        df3 = pd.read_excel(f'{over_the_folder_path}/{a[-2]} год/K6-{a[-2]}.xls', 
                        sheet_name='Раздел 2')
        df4 = pd.read_excel(f'{over_the_folder_path}/{a[-2]} год/F1-{a[-2]}.xls', 
                        sheet_name='Раздел 1')
        # достаем нужные данные по функции, добавляем эти данные в датафрейм
        # for a in article_rows.keys():
        new_row = note_data(df0, df, df2, df3, df4)
        rows.append(new_row)
        print(a[-2], rows[-1])
print('All done!')

2020 year                      2020.0
article                 109 ч. 2
total                        114
restriction                   82
suspention                     1
disqualification               0
disq_plus                     33
acquital                      18
reconciliation                16
total_category              1785
excuse (category)             79
exemption (category)          45
sent_back (category)         117
sent_down (category)          39
dtype: object
2021 year                      2021.0
article                 109 ч. 2
total                        122
restriction                   87
suspention                     4
disqualification               0
disq_plus                     24
acquital                      13
reconciliation                22
total_category              1764
excuse (category)             96
exemption (category)          47
sent_back (category)         106
sent_down (category)          22
dtype: object
2023 year                      2023.0


In [543]:
b = '2025'
c = '1-2025'
df0 = pd.read_excel(f'{over_the_folder_path}/{b} год (первое полугодие)/K7-{c}.xls')
df = pd.read_excel(f'{over_the_folder_path}/{b} год (первое полугодие)/K7-{c}.xls', 
                        sheet_name='Раздел 1')
df2 = pd.read_excel(f'{over_the_folder_path}/{b} год (первое полугодие)/K6-{c}.xls', 
                sheet_name='Раздел 1')
df3 = pd.read_excel(f'{over_the_folder_path}/{b} год (первое полугодие)/K6-{c}.xls', 
                sheet_name='Раздел 2')
df4 = pd.read_excel(f'{over_the_folder_path}/{b} год (первое полугодие)/F1-{c}.xls', 
                sheet_name='Раздел 1')

# 'article': 
df2.iloc[11, 3], #статья
# 'restriction': df.iloc[16, 8], #ограничение свободы
# 'suspention': df.iloc[16, 5], #условный срок
# 'disqualification': df.iloc[16, 13], #запрет на проф. деятельность
# 'disq_plus': df.iloc[16, 31], #запрет на проф. деятельность как доп. наказание
# 'acquital': df.iloc[16, 22], #оправдание
# 'reconciliation': df.iloc[16, 25], #примирение сторон
# 'excuse (category)': 
# df2.iloc[15, 34], #дело прекратили из-за истечения срока давности
# 'exemption (category)': 
# df3.iloc[15, 26], #освободили от наказания из-за истечения срока давности
# 'sent_back (category)': 
# df4.iloc[12, 10] #дело вернули в прокуратуру

# достаем нужные данные по функции, добавляем эти данные в датафрейм
new_row = note_data(df0, df, df2, df3, df4)
rows.append(new_row)
table = pd.DataFrame(rows)

In [544]:
table.sort_values(by='year')

,year,article,total,restriction,suspention,disqualification,disq_plus,acquital,reconciliation,total_category,excuse (category),exemption (category),sent_back (category),sent_down (category)
5,2018.0,109 ч. 2,151,115,2,0,0,39,42,1930,59,41,102,50
4,2019.0,109 ч. 2,143,99,1,0,48,20,32,1904,75,50,115,49
0,2020.0,109 ч. 2,114,82,1,0,33,18,16,1785,79,45,117,39
1,2021.0,109 ч. 2,122,87,4,0,24,13,22,1764,96,47,106,22
3,2022.0,109 ч. 2,116,74,0,0,21,12,37,1800,108,70,60,23
2,2023.0,109 ч. 2,116,80,0,0,23,8,27,1558,77,67,44,20
6,2024.0,109 ч. 2,124,80,2,0,15,9,10,1576,86,74,40,23
7,2025.0,109 ч. 2,57,44,0,0,13,9,5,595,42,23,16,9


In [545]:
table['restriction_share'] = table['restriction'] / table['total'] * 100
table['suspension_share'] = table['suspention'] / table['total'] * 100
table['disq_plus_share'] = table['disq_plus'] / table['total'] * 100
table['acquital_share'] = table['acquital'] / table['total'] * 100
table['reconciliation_share'] = table['reconciliation'] / table['total'] * 100
table['exemption_share'] = table['exemption (category)'] / table['total_category'] * 100
table['excuse_share'] = table['excuse (category)'] / table['total_category'] * 100

In [549]:
table.sort_values(by='year').to_clipboard()

In [547]:
table.to_csv('../output/obstetric_violence_sudrf.csv')

## Описательная статистика по суд. депу и ак. насилию

In [516]:
all_articles['yearDate'] = all_articles['yearDate'].astype(str)
all_articles.loc[all_articles['109'] == 'TRUE']['id'].groupby(by='yearDate').value_counts()

KeyError: 'yearDate'

# Выделяем топ слов из дел по акушерскому насилию

In [298]:
# pd.set_option('display.max_colwidth', 50)
# path = '/Users/Helen/Library/CloudStorage/GoogleDrive-elena.d.ovchinnikova@gmail.com/My Drive/6. FREELANCE STORIES/Horizontal Russia/2025 OBSTETRIC VIOLENCE/0. GITHUB'
# ff = pd.read_csv(f'{path}/childbirth_all_articles.csv')
# ff.head()

In [299]:
# from collections import Counter

# # Filter rows where 'relevant' == 'yes' and 'text' is not null
# texts = ff.loc[(ff['relevant'] == 'yes') & (ff['text'].notnull()), 'text']

# # Tokenize and count words
# all_words = []
# for t in texts:
#     all_words.extend(t.lower().split())

# word_counts = Counter(all_words)
# top_words = word_counts.most_common(100)
# print(top_words)

In [300]:
# duplicates = pd.read_csv('../output/find duplicates.csv')
# duplicates.head()

In [301]:
# # Create a column 'in_all_articles' that is True if '105_2' appears anywhere in 'all_articles'
# duplicates['in_all_articles'] = duplicates['105_2'].isin(duplicates['all_articles'].values)
# duplicates.head()

In [302]:
# duplicates['in_all_articles'].describe()